## 訓練模型

### 讀檔

In [19]:
# ------------------------------------
# 取得csv的絕對路徑
# ------------------------------------

import os
# current_dir = os.path.dirname(os.path.abspath(__file__))

current_dir: str = os.getcwd()
csv_path = os.path.join(current_dir, "Salary_Data2.csv")
print(csv_path)
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"找不到數據集檔案:{csv_path}")
   



D:\Github\20260703_ai_tvdi\backend\07_31\Salary_Data2.csv


In [20]:
import pandas as pd
import time
from pandas import DataFrame
from sklearn.preprocessing import OrdinalEncoder

data:DataFrame = pd.read_csv(csv_path)
#display(data)
# 開始的時間
start_time:float = time.time()


# ----------------------------------------------------
# 1. 建立並擬合 OrdinalEncoder (學歷：高中以下=0, 大學=1, 碩士以上=2)
# 顯式指定類別位階順序，確保高學歷對應較高數值：
# 優先階層
# oe.categories_ 會是 [['高中以下', '大學', '碩士以上']] (索引 0: 高中以下, 1: 大學, 2: 碩士以上)
# ----------------------------------------------------

oe = OrdinalEncoder(categories=[['高中以下','大學','碩士以上']])
data['EducationLevel'] = oe.fit_transform(data[['EducationLevel']])
#display(data)



In [21]:
# ----------------------------------------------------
# 2. 建立並整合 oneHotEncoder (城市：城市A, 城市B, 城市C)
# ----------------------------------------------------

from sklearn.preprocessing import OneHotEncoder
#display(data['City'].unique())

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # 為了要array 要sparse_output=false
ohe.fit(pd.DataFrame([["城市A"],["城市C"],["城市B"]], columns=["City"]))
city_encoded = ohe.transform(data[['City']])
city_cols = ohe.get_feature_names_out(['City'])
city_df = pd.DataFrame(city_encoded, columns=city_cols)
data = pd.concat([data, city_df], axis=1).drop('City', axis=1)   # 左右合併用 axis=1
#display(data)



In [27]:
# ----------------------------------------------------
# 開始訓練
# -----------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# 定義特徵欄位與目標變數
feature_names = ['YearsExperience','EducationLevel','City_城市A','City_城市B','City_城市C']
X = data [feature_names]
y = data ['Salary']

# 切分訓練集與測試集
test_size = 0.2
random_state = 76

X_train, X_test, y_train, y_test =train_test_split(
    X, y, test_size = test_size, random_state = random_state
)

# 特徵標準化(對所有待徵進行)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 關於模型
model_type = "LinearRegression "
alpha = 1.0
model_type_clean = model_type.strip()
if model_type_clean.lower() == "lasso":
    model = Lasso(alpha=alpha, random_state=random_state)
    actual_model_name = f"Lasso 迴歸(α={alpha})"
    model_type_clean="Lasso"
elif model_type_clean.lower() == "ridge":
    model = Ridge(alpha=alpha, random_state=random_state)
    actual_model_name = f"Ridge 嶺迴歸(α={alpha})"
    model_type_clean="Ridge"
else:
    model = LinearRegression()
    actual_model_name = "多元線性迴歸 (OLS)"
    model_type_clean="LinearRegression"

print(f"開始訓練 {actual_model_name} (測試集比例:{test_size}, 隨機種子:{random_state})....")
model.fit(X_train_scaled, y_train)

train_time = time.time() - start_time
display(train_time)

開始訓練 多元線性迴歸 (OLS) (測試集比例:0.2, 隨機種子:76)....


36.51315355300903

In [28]:
# ----------------------------------------------------
# 取得模型的權重,偏移值,評估值R2
# ----------------------------------------------------

r2 = model.score(X_test_scaled, y_test)

coefs = model.coef_
intercept = model.intercept_
feature_coefs = {
 name: float (coef) for name, coef  in zip(feature_names, coefs)
}
display(feature_coefs, r2)


{'YearsExperience': 5.69635898409509,
 'EducationLevel': 15.307434019228705,
 'City_城市A': 0.5656436343757925,
 'City_城市B': -3.5283993727742757,
 'City_城市C': 1.3122634452625777}

0.8462535226367868

In [24]:
# ----------------------------------------------------
# 存模型以及所有相關預處理器與元數據 (Metadata)
# ----------------------------------------------------
import joblib

model_data = {
    "model": model,
    "oe": oe,
    "ohe": ohe,
    "scaler": scaler,
    "r2": float(r2),
    "coef": [float(c) for c in coefs],
    "intercept": float(intercept),
    "feature_names": feature_names,
    "feature_coefs": feature_coefs,
    "model_type": model_type_clean,
    "alpha": float(alpha),
    "train_time": float(train_time),
    "test_size": test_size,
    "random_state": random_state
}

model_filename = os.path.join(current_dir, "salary_model.joblib")
print(f"正在轉模型、預處理器與元數據序列化並儲存至{model_filename}...")
joblib.dump(model_data, model_filename)
print("模型儲存成功!")

正在轉模型、預處理器與元數據序列化並儲存至D:\Github\20260703_ai_tvdi\backend\07_31\salary_model.joblib...
模型儲存成功!
